In [ ]:
!git clone https://github.com/Vongole25/University-Projects.git
%cd University-Projects
!pip -q install -U pip
!pip -q install -e stereoset_topic_bias

In [ ]:
import os
os.environ["HF_TOKEN"] = "hf_***REDACTED***"

In [ ]:
!llm-ss run \
  --model_id meta-llama/Meta-Llama-3-8B-Instruct \
  --subset both \
  --seed 42 \
  --dtype bf16 \
  --device_map auto \
  --batch_size 1 \
  --out stereoset_topic_bias/runs/v0_1_llama3_instruct_full_s42

In [ ]:
!llm-ss analyze --run_dir stereoset_topic_bias/runs/v0_1_llama3_instruct_full_s42 --bootstrap 1000 --seed 42
!llm-ss report  --run_dir stereoset_topic_bias/runs/v0_1_llama3_instruct_full_s42


In [ ]:
%%bash
set -e

MODEL="meta-llama/Meta-Llama-3-8B-Instruct"
BASE_OUT="stereoset_topic_bias/runs"
BOOT=1000

for S in 42 43 44; do
  OUT="${BASE_OUT}/v0_1_llama3_instruct_full_s${S}"
  echo "=== SEED ${S} | OUT=${OUT} ==="

  llm-ss run \
    --model_id "${MODEL}" \
    --subset both \
    --seed "${S}" \
    --dtype bf16 \
    --device_map auto \
    --batch_size 1 \
    --out "${OUT}"

  llm-ss analyze \
    --run_dir "${OUT}" \
    --bootstrap "${BOOT}" \
    --seed "${S}"

  llm-ss report --run_dir "${OUT}"
done


In [ ]:
import os, glob, re
import pandas as pd

RUN_GLOB = "stereoset_topic_bias/runs/v0_1_llama3_instruct_full_s*/summary_with_ci.csv"

paths = sorted(glob.glob(RUN_GLOB))
assert paths, f"No files matched: {RUN_GLOB}"

def seed_from_path(p):
    m = re.search(r"_s(\d+)", p)
    return int(m.group(1)) if m else None

dfs = []
for p in paths:
    df = pd.read_csv(p)
    df["seed"] = seed_from_path(p)
    df["run_dir"] = os.path.dirname(p)
    dfs.append(df)

all_sum = pd.concat(dfs, ignore_index=True)

# 편향 강도(방향 제외)도 같이 계산
all_sum["bias_strength"] = (all_sum["ss"] - 50).abs()

# seed별 원본 요약(원하면 확인)
print("Loaded runs:", all_sum["run_dir"].nunique(), "seeds:", sorted(all_sum["seed"].unique()))

agg = (all_sum
       .groupby(["subset","domain"], as_index=False)
       .agg(
           n_mean=("n","mean"),
           ss_mean=("ss","mean"),
           ss_std=("ss","std"),
           bias_strength_mean=("bias_strength","mean"),
           icat_mean=("icat","mean"),
           icat_std=("icat","std"),
       )
      )

agg = agg.sort_values(["subset","domain"])
agg


In [ ]:
agg.to_csv("instruct_seeds_aggregate.csv", index=False)
print("saved: instruct_seeds_aggregate.csv")


In [ ]:
import pandas as pd, glob, os, re

DELTA_GLOB = "stereoset_topic_bias/runs/v0_1_llama3_instruct_full_s*/delta_ss.csv"
paths = sorted(glob.glob(DELTA_GLOB))
assert paths, f"No files matched: {DELTA_GLOB}"

def seed_from_path(p):
    m = re.search(r"_s(\d+)", p)
    return int(m.group(1)) if m else None

dfs = []
for p in paths:
    df = pd.read_csv(p)
    df["seed"] = seed_from_path(p)
    df["run_dir"] = os.path.dirname(p)
    dfs.append(df)

all_delta = pd.concat(dfs, ignore_index=True)

# pair key 정규화(순서 유지: domain_a-domain_b 그대로)
all_delta["pair"] = all_delta["domain_a"].astype(str) + " - " + all_delta["domain_b"].astype(str)

stable = (all_delta
          .groupby(["subset","pair"], as_index=False)
          .agg(
              seeds=("seed","nunique"),
              sig_count=("significant","sum"),
              delta_mean=("delta_ss","mean"),
              delta_std=("delta_ss","std"),
              ci_low_mean=("ci_low","mean"),
              ci_high_mean=("ci_high","mean"),
          )
         )

# 3개 시드 중 2개 이상에서 유의한 것부터 위로
stable = stable.sort_values(["sig_count","subset"], ascending=[False, True])
stable


In [ ]:
stable.to_csv("instruct_seeds_delta_stability.csv", index=False)
print("saved: instruct_seeds_delta_stability.csv")


In [ ]:
!zip -r instruct_seed_summary.zip instruct_seeds_aggregate.csv instruct_seeds_delta_stability.csv


In [ ]:
%%bash
set -e

MODEL="meta-llama/Meta-Llama-3-8B"
BASE_OUT="stereoset_topic_bias/runs"
BOOT=1000

for S in 42 43 44; do
  OUT="${BASE_OUT}/v0_1_llama3_base_full_s${S}"
  echo "=== BASE | SEED ${S} ==="

  llm-ss run \
    --model_id "${MODEL}" \
    --subset both \
    --seed "${S}" \
    --dtype bf16 \
    --device_map auto \
    --batch_size 1 \
    --out "${OUT}"

  llm-ss analyze --run_dir "${OUT}" --bootstrap "${BOOT}" --seed "${S}"
done


In [ ]:
import glob, os, re
import pandas as pd

def load_runs(glob_pat, tag):
    paths = sorted(glob.glob(glob_pat))
    dfs = []
    for p in paths:
        df = pd.read_csv(p)
        m = re.search(r"_s(\d+)", p)
        df["seed"] = int(m.group(1)) if m else None
        df["tag"] = tag
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

ins = load_runs("stereoset_topic_bias/runs/v0_1_llama3_instruct_full_s*/summary_with_ci.csv", "instruct")
bas = load_runs("stereoset_topic_bias/runs/v0_1_llama3_base_full_s*/summary_with_ci.csv", "base")

all_df = pd.concat([ins, bas], ignore_index=True).dropna(subset=["seed"])
all_df["bias_strength"] = (all_df["ss"] - 50).abs()

agg = (all_df.groupby(["tag","subset","domain"], as_index=False)
       .agg(ss_mean=("ss","mean"),
            ss_std=("ss","std"),
            bias_strength_mean=("bias_strength","mean"),
            icat_mean=("icat","mean"),
            icat_std=("icat","std"),
            n_mean=("n","mean")))

# instruct - base 차이(SS 기준)
pivot = agg.pivot_table(index=["subset","domain"], columns="tag", values="ss_mean")
pivot["instruct_minus_base"] = pivot["instruct"] - pivot["base"]
pivot = pivot.reset_index().sort_values(["subset","domain"])
pivot


In [ ]:
import os, glob, re
import pandas as pd

# ✅ run 폴더 패턴 (네가 사용한 이름 그대로)
INS_GLOB  = "stereoset_topic_bias/runs/v0_1_llama3_instruct_full_s*/summary_with_ci.csv"
BASE_GLOB = "stereoset_topic_bias/runs/v0_1_llama3_base_full_s*/summary_with_ci.csv"

def seed_from_path(p: str):
    m = re.search(r"_s(\d+)", p)
    return int(m.group(1)) if m else None

def load_seed_summaries(glob_pat: str, tag: str) -> pd.DataFrame:
    paths = sorted(glob.glob(glob_pat))
    if not paths:
        raise FileNotFoundError(f"No files matched pattern: {glob_pat}")

    dfs = []
    for p in paths:
        df = pd.read_csv(p)
        # 필요한 컬럼 최소만 유지 (혹시 이름이 다르면 여기서 확인 가능)
        required = {"subset", "domain", "n", "ss"}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"{p} is missing columns: {missing}. Available: {list(df.columns)}")

        df = df[["subset", "domain", "n", "ss"]].copy()
        df["seed"] = seed_from_path(p)
        df["tag"] = tag
        df["run_dir"] = os.path.dirname(p)
        df["bias_strength"] = (df["ss"] - 50).abs()
        dfs.append(df)

    out = pd.concat(dfs, ignore_index=True)
    out = out.dropna(subset=["seed"])
    return out

def aggregate_across_seeds(df: pd.DataFrame, tag: str) -> pd.DataFrame:
    # seed 평균/표준편차 (deterministic이면 std=0이 정상)
    agg = (df.groupby(["subset", "domain"], as_index=False)
             .agg(
                 n_mean=("n", "mean"),
                 ss_mean=("ss", "mean"),
                 ss_std=("ss", "std"),
                 bias_strength_mean=("bias_strength", "mean"),
                 bias_strength_std=("bias_strength", "std"),
                 seeds=("seed", "nunique"),
             ))
    # 컬럼 이름에 tag 붙이기
    rename = {c: f"{c}_{tag}" for c in agg.columns if c not in ("subset", "domain")}
    agg = agg.rename(columns=rename)
    return agg

# 1) 로드
ins_raw  = load_seed_summaries(INS_GLOB, "instruct")
base_raw = load_seed_summaries(BASE_GLOB, "base")

print("Loaded instruct seeds:", sorted(ins_raw["seed"].unique()), "| runs:", ins_raw["run_dir"].nunique())
print("Loaded base seeds    :", sorted(base_raw["seed"].unique()), "| runs:", base_raw["run_dir"].nunique())

# 2) seed 집계
ins_agg  = aggregate_across_seeds(ins_raw, "instruct")
base_agg = aggregate_across_seeds(base_raw, "base")

# 3) merge + shift 계산
merged = ins_agg.merge(base_agg, on=["subset", "domain"], how="inner")

merged["ss_shift_instruct_minus_base"] = merged["ss_mean_instruct"] - merged["ss_mean_base"]
merged["bias_strength_shift"] = merged["bias_strength_mean_instruct"] - merged["bias_strength_mean_base"]

# 보기 좋게 정렬
merged = merged.sort_values(["subset", "domain"]).reset_index(drop=True)

# 4) 저장
out_path = "instruct_vs_base_ss_shift.csv"
merged.to_csv(out_path, index=False)
print("Saved:", out_path)

# 5) 화면 출력(핵심 컬럼만)
cols = [
    "subset","domain",
    "n_mean_instruct","ss_mean_instruct","ss_std_instruct",
    "n_mean_base","ss_mean_base","ss_std_base",
    "ss_shift_instruct_minus_base",
    "bias_strength_mean_instruct","bias_strength_mean_base","bias_strength_shift",
]
display(merged[cols])

# 6) 변화량 큰 순(top 10)
top = merged.copy()
top["abs_ss_shift"] = top["ss_shift_instruct_minus_base"].abs()
top = top.sort_values("abs_ss_shift", ascending=False)
print("\nTop SS shifts (abs):")
display(top[["subset","domain","ss_mean_instruct","ss_mean_base","ss_shift_instruct_minus_base","bias_strength_shift"]].head(10))

# 7) subset별 '가중 평균 SS' 이동량 (도메인 n_mean으로 가중)
def weighted_mean(group, value_col, weight_col):
    w = group[weight_col].astype(float)
    v = group[value_col].astype(float)
    return (v * w).sum() / w.sum()

overall = []
for subset in merged["subset"].unique():
    g = merged[merged["subset"] == subset]
    ss_ins = weighted_mean(g, "ss_mean_instruct", "n_mean_instruct")
    ss_bas = weighted_mean(g, "ss_mean_base", "n_mean_base")
    overall.append({
        "subset": subset,
        "weighted_ss_instruct": ss_ins,
        "weighted_ss_base": ss_bas,
        "weighted_ss_shift": ss_ins - ss_bas
    })

overall_df = pd.DataFrame(overall).sort_values("subset")
overall_df.to_csv("instruct_vs_base_weighted_ss_shift.csv", index=False)
print("\nSaved: instruct_vs_base_weighted_ss_shift.csv")
display(overall_df)


In [ ]:
import glob, os, re
import pandas as pd

def seed_from_path(p: str):
    m = re.search(r"_s(\d+)", p)
    return int(m.group(1)) if m else None

def load_delta(glob_pat: str, tag: str) -> pd.DataFrame:
    paths = sorted(glob.glob(glob_pat))
    if not paths:
        raise FileNotFoundError(f"No files matched: {glob_pat}")
    dfs = []
    for p in paths:
        df = pd.read_csv(p)
        df["seed"] = seed_from_path(p)
        df["tag"] = tag
        df["pair"] = df["domain_a"].astype(str) + " - " + df["domain_b"].astype(str)
        dfs.append(df)
    out = pd.concat(dfs, ignore_index=True).dropna(subset=["seed"])
    return out

ins = load_delta("stereoset_topic_bias/runs/v0_1_llama3_instruct_full_s*/delta_ss.csv", "instruct")
bas = load_delta("stereoset_topic_bias/runs/v0_1_llama3_base_full_s*/delta_ss.csv", "base")

# seed 평균(결정론이면 std=0)
def agg(df):
    return (df.groupby(["tag","subset","pair"], as_index=False)
              .agg(delta_mean=("delta_ss","mean"),
                   delta_std=("delta_ss","std"),
                   sig_count=("significant","sum"),
                   seeds=("seed","nunique")))

A = agg(ins)
B = agg(bas)

comp = (A.merge(B, on=["subset","pair"], suffixes=("_instruct","_base"))
          .assign(delta_shift=lambda x: x["delta_mean_instruct"] - x["delta_mean_base"],
                  sig_changed=lambda x: (x["sig_count_instruct"] != x["sig_count_base"]))
       )

comp = comp.sort_values(["sig_changed","subset","pair"], ascending=[False, True, True])
comp.to_csv("instruct_vs_base_delta_compare.csv", index=False)
print("saved: instruct_vs_base_delta_compare.csv")

# 유의성 변화 있는 것부터 보기
display(comp[["subset","pair",
              "delta_mean_instruct","sig_count_instruct",
              "delta_mean_base","sig_count_base",
              "delta_shift","sig_changed"]].head(30))


In [ ]:
import pandas as pd

# 파일 경로 (필요하면 수정)
PATH = "instruct_vs_base_delta_compare.csv"

df = pd.read_csv(PATH)

# 방어: 필요한 컬럼 확인
needed = {
    "subset","pair",
    "delta_mean_instruct","sig_count_instruct",
    "delta_mean_base","sig_count_base",
    "delta_shift","sig_changed"
}
missing = needed - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}\nAvailable: {list(df.columns)}")

# 편의 컬럼
df["abs_delta_shift"] = df["delta_shift"].abs()

# 1) 절댓값 변화량 TOP 5
top5_abs = df.sort_values("abs_delta_shift", ascending=False).head(5)
print("\n=== TOP 5 | abs(delta_shift) ===")
display(top5_abs[[
    "subset","pair",
    "delta_mean_base","sig_count_base",
    "delta_mean_instruct","sig_count_instruct",
    "delta_shift","abs_delta_shift","sig_changed"
]])

top5_abs.to_csv("delta_shift_top5_abs.csv", index=False)

# 2) 유의성 변화 있었던 쌍(전부 출력)
changed = df[df["sig_changed"] == True].sort_values("abs_delta_shift", ascending=False)
print("\n=== Significant status changed (sig_changed=True) ===")
display(changed[[
    "subset","pair",
    "delta_mean_base","sig_count_base",
    "delta_mean_instruct","sig_count_instruct",
    "delta_shift","abs_delta_shift"
]])

changed.to_csv("delta_shift_sig_changed.csv", index=False)

# 3) (선택) Instruct에서 3/3 유의한 쌍 중 변화량 TOP 5
ins_sig = df[df["sig_count_instruct"] == 3].sort_values("abs_delta_shift", ascending=False).head(5)
print("\n=== TOP 5 | abs(delta_shift) among instruct significant (3/3) ===")
display(ins_sig[[
    "subset","pair",
    "delta_mean_base","sig_count_base",
    "delta_mean_instruct","sig_count_instruct",
    "delta_shift","abs_delta_shift"
]])

ins_sig.to_csv("delta_shift_top5_instruct_sig.csv", index=False)

print("\nSaved CSVs: delta_shift_top5_abs.csv, delta_shift_sig_changed.csv, delta_shift_top5_instruct_sig.csv")


In [ ]:
import pandas as pd

# seed 하나만 대표로 써도 되고(결정론이라 거의 동일), 42/43/44 모두 확인해도 됨
p = "stereoset_topic_bias/runs/v0_1_llama3_base_full_s42/summary_with_ci.csv"
df = pd.read_csv(p)

# SS + CI만 보기 좋게
out = df[["subset","domain","n","ss","ss_ci_low","ss_ci_high"]].sort_values(["subset","domain"])
print(out.to_string(index=False))
